# Day 2 · Track B — Lab 1
# B1: LangGraph & Microsoft Agent Framework · B2: MCP & A2A Integration

This notebook is self-contained: it runs with **only the Python standard library** by default
(a lightweight `MiniGraph` engine that mimics LangGraph's `StateGraph` API), and includes
**optional real-library cells** you can switch on with `pip install` if you want to run it
against actual LangGraph / Microsoft Agent Framework packages.

**Contents**
1. B1a — Build a multi-node agent graph (router → specialist → responder) with a mock `StateGraph`
2. B1b — Same graph, real `langgraph` version (optional, requires install)
3. B1c — Microsoft Agent Framework style: role-based agents + orchestrator (mock, install-optional)
4. B2a — MCP (Model Context Protocol): a minimal in-process MCP-style tool server + client
5. B2b — A2A (Agent-to-Agent): message bus so two independent agents negotiate a task
6. B2c — Putting B1 + B2 together: a LangGraph agent that calls an MCP tool and hands off via A2A

Run cells top to bottom. No API keys required for the default path.


## Setup — optional installs
Uncomment and run if you want the *real* libraries instead of the mock engine below.

In [ ]:
# Optional: uncomment to use real LangGraph / OpenAI-compatible agents
# %pip install langgraph langchain-core --quiet
# %pip install microsoft-agent-framework --quiet   # package name may vary by release; see MS docs

import sys, json, uuid, time, random, textwrap
from dataclasses import dataclass, field
from typing import Callable, Dict, Any, List, Optional

print("Python:", sys.version.split()[0])


## B1a — Mock `StateGraph` engine

LangGraph's core idea: a **graph of nodes** that read/write a shared `state` dict, with
**edges** (including conditional edges) deciding what runs next. We reproduce the minimal
mechanics here so the lab runs with zero installs.

In [ ]:
class MiniGraph:
    """A tiny reproduction of LangGraph's StateGraph mechanics."""

    def __init__(self):
        self.nodes: Dict[str, Callable[[dict], dict]] = {}
        self.edges: Dict[str, str] = {}                 # simple fixed edges
        self.cond_edges: Dict[str, Callable[[dict], str]] = {}  # conditional edges
        self.entry: Optional[str] = None
        self.END = "__END__"

    def add_node(self, name: str, fn: Callable[[dict], dict]):
        self.nodes[name] = fn
        return self

    def set_entry(self, name: str):
        self.entry = name
        return self

    def add_edge(self, src: str, dst: str):
        self.edges[src] = dst
        return self

    def add_conditional_edge(self, src: str, router: Callable[[dict], str]):
        self.cond_edges[src] = router
        return self

    def run(self, state: dict, max_steps: int = 25, trace: bool = True) -> dict:
        node = self.entry
        steps = 0
        while node != self.END and steps < max_steps:
            if trace:
                print(f"[step {steps}] -> node: {node}")
            fn = self.nodes[node]
            state = fn(state)
            if node in self.cond_edges:
                node = self.cond_edges[node](state)
            elif node in self.edges:
                node = self.edges[node]
            else:
                node = self.END
            steps += 1
        if trace:
            print(f"[step {steps}] -> END")
        state["_steps"] = steps
        return state


### Build a 3-node agent: `router → specialist → responder`
This mirrors a common LangGraph pattern: classify intent, route to a specialist node, then format the final answer.

In [ ]:
def router_node(state: dict) -> dict:
    text = state["input"].lower()
    if any(k in text for k in ["invoice", "remittance", "payment", "match"]):
        state["route"] = "finance_specialist"
    elif any(k in text for k in ["voice", "call", "speech"]):
        state["route"] = "voice_specialist"
    else:
        state["route"] = "general_specialist"
    return state

def finance_specialist(state: dict) -> dict:
    state["draft"] = f"[Finance] Parsed request: '{state['input']}'. Suggested action: run 3-way match."
    return state

def voice_specialist(state: dict) -> dict:
    state["draft"] = f"[Voice] Transcribed intent handled for: '{state['input']}'."
    return state

def general_specialist(state: dict) -> dict:
    state["draft"] = f"[General] Handled generic query: '{state['input']}'."
    return state

def responder(state: dict) -> dict:
    state["output"] = f"AGENT RESPONSE: {state['draft']}"
    return state

def route_selector(state: dict) -> str:
    return state["route"]

graph = MiniGraph()
graph.add_node("router", router_node)
graph.add_node("finance_specialist", finance_specialist)
graph.add_node("voice_specialist", voice_specialist)
graph.add_node("general_specialist", general_specialist)
graph.add_node("responder", responder)

graph.set_entry("router")
graph.add_conditional_edge("router", route_selector)
graph.add_edge("finance_specialist", "responder")
graph.add_edge("voice_specialist", "responder")
graph.add_edge("general_specialist", "responder")
graph.add_edge("responder", graph.END)

for query in [
    "Please match this invoice against the remittance advice",
    "Can you handle this incoming voice call transcript?",
    "What is the weather today?",
]:
    result = graph.run({"input": query})
    print(">>", result["output"])
    print("-" * 60)


## B1b — Real LangGraph version (optional)

Same graph, expressed with the actual `langgraph` package. Requires:
```bash
pip install langgraph langchain-core
```
This cell is wrapped in a try/except so the notebook still runs top-to-bottom even if you
haven't installed the package.

In [ ]:
try:
    from langgraph.graph import StateGraph, END
    from typing import TypedDict

    class AgentState(TypedDict):
        input: str
        route: str
        draft: str
        output: str

    def router_node_lg(state: AgentState):
        text = state["input"].lower()
        if any(k in text for k in ["invoice", "remittance", "payment", "match"]):
            return {"route": "finance_specialist"}
        elif any(k in text for k in ["voice", "call", "speech"]):
            return {"route": "voice_specialist"}
        return {"route": "general_specialist"}

    def finance_specialist_lg(state: AgentState):
        return {"draft": f"[Finance] {state['input']}"}

    def voice_specialist_lg(state: AgentState):
        return {"draft": f"[Voice] {state['input']}"}

    def general_specialist_lg(state: AgentState):
        return {"draft": f"[General] {state['input']}"}

    def responder_lg(state: AgentState):
        return {"output": f"AGENT RESPONSE: {state['draft']}"}

    g = StateGraph(AgentState)
    g.add_node("router", router_node_lg)
    g.add_node("finance_specialist", finance_specialist_lg)
    g.add_node("voice_specialist", voice_specialist_lg)
    g.add_node("general_specialist", general_specialist_lg)
    g.add_node("responder", responder_lg)

    g.set_entry_point("router")
    g.add_conditional_edges("router", lambda s: s["route"], {
        "finance_specialist": "finance_specialist",
        "voice_specialist": "voice_specialist",
        "general_specialist": "general_specialist",
    })
    g.add_edge("finance_specialist", "responder")
    g.add_edge("voice_specialist", "responder")
    g.add_edge("general_specialist", "responder")
    g.add_edge("responder", END)

    app = g.compile()
    out = app.invoke({"input": "Please match this invoice against the remittance advice"})
    print(out["output"])

except ImportError:
    print("langgraph not installed — skipping real-library demo.")
    print("Install with:  pip install langgraph langchain-core")


## B1c — Microsoft Agent Framework style: role-based agents + orchestrator

The Microsoft Agent Framework (successor combining AutoGen + Semantic Kernel agent patterns)
centers on **named agents with roles**, an **orchestrator**, and **turn-based handoff**.
We mock the essential shape below; swap in the real SDK client the same way as B1b if you
have access to it (`pip install agent-framework` — check current MS docs for exact package name).

In [ ]:
@dataclass
class Agent:
    name: str
    role: str
    handle: Callable[[str, dict], str]

class Orchestrator:
    """Round-robins a task between role-based agents until one marks it DONE."""
    def __init__(self, agents: List[Agent], max_turns: int = 6):
        self.agents = agents
        self.max_turns = max_turns

    def run(self, task: str) -> List[str]:
        transcript = []
        context = {"task": task}
        for turn in range(self.max_turns):
            agent = self.agents[turn % len(self.agents)]
            msg = agent.handle(task, context)
            transcript.append(f"{agent.name} ({agent.role}): {msg}")
            if "DONE" in msg:
                break
        return transcript

def planner_fn(task, ctx):
    if "plan" not in ctx:
        ctx["plan"] = ["extract_fields", "validate", "match"]
        return f"Plan created: {ctx['plan']}"
    return f"Plan already in progress: {ctx['plan']}"

def executor_fn(task, ctx):
    plan = ctx.get("plan", [])
    if plan:
        step = plan.pop(0)
        ctx.setdefault("done_steps", []).append(step)
        return f"Executed step '{step}'. Remaining: {plan}"
    return "DONE — all steps executed."

planner = Agent("Planner", "planning", planner_fn)
executor = Agent("Executor", "execution", executor_fn)

orch = Orchestrator([planner, executor], max_turns=8)
for line in orch.run("Process remittance file and match to invoices"):
    print(line)


---
## B2a — MCP (Model Context Protocol): minimal tool server + client

Real MCP runs over stdio/SSE with JSON-RPC between a host process and a server process.
For a runnable-anywhere lab, we simulate the **protocol shape** (tool discovery → tool call →
structured result) in-process. The message format mirrors real MCP JSON-RPC payloads so the
concepts transfer directly.

In [ ]:
class MCPServer:
    """Simulates an MCP server exposing tools, mirroring MCP's tools/list + tools/call shape."""
    def __init__(self, name: str):
        self.name = name
        self.tools: Dict[str, Callable[[dict], dict]] = {}
        self.schemas: Dict[str, dict] = {}

    def register_tool(self, name: str, schema: dict, fn: Callable[[dict], dict]):
        self.tools[name] = fn
        self.schemas[name] = schema
        return self

    def handle_request(self, request: dict) -> dict:
        method = request.get("method")
        if method == "tools/list":
            return {"jsonrpc": "2.0", "id": request["id"],
                     "result": {"tools": [{"name": n, "schema": s} for n, s in self.schemas.items()]}}
        if method == "tools/call":
            name = request["params"]["name"]
            args = request["params"].get("arguments", {})
            if name not in self.tools:
                return {"jsonrpc": "2.0", "id": request["id"],
                         "error": {"code": -32601, "message": f"Unknown tool: {name}"}}
            result = self.tools[name](args)
            return {"jsonrpc": "2.0", "id": request["id"], "result": result}
        return {"jsonrpc": "2.0", "id": request["id"], "error": {"code": -32601, "message": "Unknown method"}}


class MCPClient:
    def __init__(self, server: MCPServer):
        self.server = server
        self._id = 0

    def _next_id(self):
        self._id += 1
        return self._id

    def list_tools(self):
        req = {"jsonrpc": "2.0", "id": self._next_id(), "method": "tools/list"}
        return self.server.handle_request(req)["result"]["tools"]

    def call_tool(self, name: str, arguments: dict):
        req = {"jsonrpc": "2.0", "id": self._next_id(),
               "method": "tools/call", "params": {"name": name, "arguments": arguments}}
        resp = self.server.handle_request(req)
        if "error" in resp:
            raise RuntimeError(resp["error"])
        return resp["result"]


# --- Define a couple of finance-domain MCP tools ---
def tool_extract_remittance(args: dict) -> dict:
    raw = args.get("raw_text", "")
    # naive extraction demo
    import re
    amount = re.search(r"\$([0-9,]+\.?[0-9]*)", raw)
    invoice = re.search(r"INV-?(\d+)", raw, re.I)
    return {
        "amount": amount.group(1) if amount else None,
        "invoice_id": f"INV-{invoice.group(1)}" if invoice else None,
        "source_text": raw,
    }

def tool_match_invoice(args: dict) -> dict:
    invoice_amount = args.get("invoice_amount")
    remit_amount = args.get("remit_amount")
    matched = invoice_amount is not None and invoice_amount == remit_amount
    return {"matched": matched, "confidence": 1.0 if matched else 0.0}

server = MCPServer("finance-tools")
server.register_tool(
    "extract_remittance",
    {"input": {"raw_text": "str"}, "output": {"amount": "str", "invoice_id": "str"}},
    tool_extract_remittance,
)
server.register_tool(
    "match_invoice",
    {"input": {"invoice_amount": "float", "remit_amount": "float"}, "output": {"matched": "bool"}},
    tool_match_invoice,
)

client = MCPClient(server)
print("Discovered tools:")
for t in client.list_tools():
    print(" -", t["name"], t["schema"])

extraction = client.call_tool("extract_remittance", {
    "raw_text": "Payment for INV-10432, amount $1,250.00 received via wire."
})
print("\nExtraction result:", extraction)

match = client.call_tool("match_invoice", {"invoice_amount": "1,250.00", "remit_amount": extraction["amount"]})
print("Match result:", match)


### Real MCP note
To run this against the **actual** MCP Python SDK instead of the mock:
```bash
pip install mcp
```
Then a server is defined with `@server.tool()` decorators and run over stdio, and a client
connects via `mcp.client.stdio.stdio_client`. The request/response shapes above (`tools/list`,
`tools/call`, JSON-RPC envelope) match the real protocol, so the mental model transfers directly.

## B2b — A2A (Agent-to-Agent): message bus + negotiation

A2A is about **independent agents** (possibly different frameworks/vendors) exchanging
structured task messages — one agent *delegates* a task to another and gets a result back,
without either needing to know the other's internals. We simulate this with a simple
in-memory bus and two independent agent objects.

In [ ]:
@dataclass
class A2AMessage:
    id: str
    sender: str
    receiver: str
    performative: str   # "request" | "inform" | "refuse" | "propose"
    content: dict

class A2ABus:
    def __init__(self):
        self.agents: Dict[str, "A2AAgent"] = {}
        self.log: List[A2AMessage] = []

    def register(self, agent: "A2AAgent"):
        self.agents[agent.name] = agent
        agent.bus = self

    def send(self, msg: A2AMessage):
        self.log.append(msg)
        receiver = self.agents[msg.receiver]
        return receiver.receive(msg)

class A2AAgent:
    def __init__(self, name: str, capabilities: List[str]):
        self.name = name
        self.capabilities = capabilities
        self.bus: Optional[A2ABus] = None

    def request(self, receiver: str, task: str, content: dict) -> A2AMessage:
        msg = A2AMessage(str(uuid.uuid4())[:8], self.name, receiver, "request",
                          {"task": task, **content})
        return self.bus.send(msg)

    def receive(self, msg: A2AMessage) -> A2AMessage:
        task = msg.content.get("task")
        if task not in self.capabilities:
            return A2AMessage(str(uuid.uuid4())[:8], self.name, msg.sender, "refuse",
                               {"reason": f"{self.name} cannot perform '{task}'"})
        result = self._perform(task, msg.content)
        return A2AMessage(str(uuid.uuid4())[:8], self.name, msg.sender, "inform", result)

    def _perform(self, task: str, content: dict) -> dict:
        raise NotImplementedError


class IngestionAgent(A2AAgent):
    def __init__(self):
        super().__init__("IngestionAgent", ["ingest_file"])

    def _perform(self, task, content):
        return {"status": "ingested", "records": 3, "source": content.get("source")}


class MatchingAgent(A2AAgent):
    def __init__(self):
        super().__init__("MatchingAgent", ["match_records"])

    def _perform(self, task, content):
        return {"status": "matched", "matched_count": 2, "unmatched_count": 1}


bus = A2ABus()
ingestion = IngestionAgent()
matching = MatchingAgent()
bus.register(ingestion)
bus.register(matching)

reply1 = ingestion.request("MatchingAgent", "match_records", {"records": 3})
print("MatchingAgent replied:", reply1)

reply2 = matching.request("IngestionAgent", "ingest_file", {"source": "erp_export.csv"})
print("IngestionAgent replied:", reply2)

reply3 = ingestion.request("MatchingAgent", "unsupported_task", {})
print("Refusal example:", reply3)

print("\nFull message log:")
for m in bus.log:
    print(" ", m)


### Real A2A note
Google's A2A protocol standardizes this over HTTP with an **AgentCard** (capability
discovery document, like `/.well-known/agent.json`), `tasks/send`, and `tasks/sendSubscribe`
for streaming. The request/inform/refuse pattern above is the same negotiation shape,
just transport-simplified for the lab.

## B2c — Putting it together: LangGraph agent → MCP tool → A2A handoff

A single node in our `MiniGraph` calls an MCP tool for extraction, then hands off the
matching step to another agent over the A2A bus.

In [ ]:
def extract_node(state: dict) -> dict:
    result = client.call_tool("extract_remittance", {"raw_text": state["input"]})
    state["extraction"] = result
    return state

def handoff_to_matching_node(state: dict) -> dict:
    reply = ingestion.request("MatchingAgent", "match_records",
                               {"records": 1, "invoice_id": state["extraction"]["invoice_id"]})
    state["match_result"] = reply.content
    return state

def final_node(state: dict) -> dict:
    state["output"] = (f"Extracted {state['extraction']} -> "
                        f"A2A match result: {state['match_result']}")
    return state

combo = MiniGraph()
combo.add_node("extract", extract_node)
combo.add_node("handoff", handoff_to_matching_node)
combo.add_node("final", final_node)
combo.set_entry("extract")
combo.add_edge("extract", "handoff")
combo.add_edge("handoff", "final")
combo.add_edge("final", combo.END)

out = combo.run({"input": "Payment for INV-98211, amount $4,500.00 received via ACH."})
print("\nFINAL:", out["output"])


---
### Exercises
1. Add a third specialist node in B1a for "compliance" queries and route to it.
2. Register a third MCP tool (`tool_flag_duplicate`) and call it from the combo graph.
3. Extend `A2AAgent` with a `propose`/`accept` negotiation step before `inform`.
